## This is the Notebook for the Algorithm to be submitted by Team Merge Error

In [ ]:
import pandas as pd
import os
from datetime import timedelta

In [ ]:
# 1. Aggregate hourly load from reefer_release.csv
file_path_reefer = '../participant_package/participant_package/reefer_release/reefer_release.csv'
df_reefer = pd.read_csv(file_path_reefer, sep=';', decimal=',')

df_reefer['EventTime'] = pd.to_datetime(df_reefer['EventTime'])
df_reefer['Hour'] = df_reefer['EventTime'].dt.floor('H')
hourly_load_w = df_reefer.groupby('Hour')['AvPowerCons'].sum()
df_hourly = pd.DataFrame(hourly_load_w / 1000.0).rename(columns={'AvPowerCons': 'load_kw'})

In [ ]:
# 2. Load target hours
file_path_targets = '../participant_package/participant_package/target_timestamps.csv'
df_targets = pd.read_csv(file_path_targets)
df_targets['timestamp_utc'] = pd.to_datetime(df_targets['timestamp_utc'].str.replace('Z', ''))

In [ ]:
# 3. Prediction fallback logic using pandas
def predict_power(row):
    hour = row['timestamp_utc']
    week_ago = hour - timedelta(hours=168)
    day_ago = hour - timedelta(hours=24)
    
    has_week_ago = week_ago in df_hourly.index
    has_day_ago = day_ago in df_hourly.index
    
    if has_week_ago and has_day_ago:
        return 0.7 * df_hourly.loc[day_ago, 'load_kw'] + 0.3 * df_hourly.loc[week_ago, 'load_kw']
    if has_day_ago:
        return df_hourly.loc[day_ago, 'load_kw']
    if has_week_ago:
        return df_hourly.loc[week_ago, 'load_kw']
    return 0.0

df_targets['pred_power_kw'] = df_targets.apply(predict_power, axis=1)
df_targets['pred_power_kw'] = df_targets['pred_power_kw'].round(6)

# 4. Add pred_p90_kw
df_targets['pred_p90_kw'] = (df_targets['pred_power_kw'] * 1.10).round(6)

# Convert timestamp_utc back to ISO format string ending with Z
df_targets['timestamp_utc'] = df_targets['timestamp_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

In [ ]:
# 5. Output to predictions.csv
output_dir = '../participant_package/participant_package/starter/output'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'predictions.csv')
df_targets.to_csv(output_path, index=False)

df_targets.head()